# HiProbCBM - Analisis tambahan: uncertainty dan subkonsep hasil discovery

Notebook ini **tidak melatih** apa pun. Untuk kedelapan skenario dan tiga seed, ia memuat `stage2_best.pth` yang sudah ada lalu menghitung:

1. **Uncertainty (prosedur ProbCBM)**: uji cutout patch 64 (`run_cutout_test`), 10 kelompok uncertainty vs akurasi (Fig. 10), dan urutan intervensi uncertainty vs acak dari `intervention_curves.json` (Fig. 11, protokol CUB). Uncertainty konsep diukur dari prediktor induk langsung dan dari variance hasil propagasi subkonsep. Uji cutout dilewati untuk PseudoKitchens karena inputnya cache fitur CLIP.
2. **Subkonsep hasil discovery (prosedur HiCEM, PseudoKitchens)**: pencocokan ke ingredient (ROC-AUC train > 0.7), rata-rata ROC-AUC test (Table 1), dan intervensi subkonsep (Fig. 4).

Hasil per seed: `uncertainty_analysis.json` dan `discovered_subconcepts.json` di folder run. Ringkasan: `train_log/analisis_tambahan.csv`.

Jalankan semua sel dari atas ke bawah pada runtime GPU T4 (VPN dimatikan).


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import torch

DRIVE_ROOT = Path("/content/drive/MyDrive/TESIS AZHIM")
assert DRIVE_ROOT.is_dir(), f"Folder tesis tidak ditemukan: {DRIVE_ROOT}"
print("Drive siap:", DRIVE_ROOT)


## 2. Clone/perbarui repo dan instal dependensi (termasuk CLIP)

In [ ]:
%cd /content
!test -d HiProbCBM || git clone https://github.com/Azhimanich/HiProbCBM.git
%cd /content/HiProbCBM
!git pull --ff-only
!pip install -e ".[clip]" -q


## 3. Siapkan CUB dan PseudoKitchens dari Drive

In [ ]:
import os
import pickle

DRIVE_CUB_ROOT = DRIVE_ROOT / "CUB_200_2011"
assert (DRIVE_CUB_ROOT / "images").is_dir(), f"Folder CUB tidak ditemukan: {DRIVE_CUB_ROOT}"
metadata = DRIVE_CUB_ROOT / "metadata"
needed = [metadata / name for name in ("train.pkl", "val.pkl", "test.pkl")]
assert all(path.is_file() for path in needed), "metadata/train.pkl, val.pkl, test.pkl harus ada di Drive."

# Normalisasi ini idempotent: hanya menulis ulang pkl bila masih ada prefix lama.
marker = "CUB_200_2011/images/"
for path in needed:
    with path.open("rb") as handle:
        records = pickle.load(handle)
    changed = sum(marker in record["img_path"] for record in records)
    if changed:
        for record in records:
            if marker in record["img_path"]:
                record["img_path"] = record["img_path"].split(marker, 1)[1]
        with path.open("wb") as handle:
            pickle.dump(records, handle)
        print(f"Dinormalisasi: {path.name} ({changed} path)")
    else:
        print(f"OK: {path.name} ({len(records)} record)")

WORK_ROOT = Path("/content/HiProbCBM/datasets/CUB_200_2011")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_CUB_ROOT.resolve(), (
        f"Symlink CUB tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_CUB_ROOT, WORK_ROOT, target_is_directory=True)
print("CUB siap:", WORK_ROOT, "->", DRIVE_CUB_ROOT)


In [ ]:
import os

DRIVE_KITCHENS_ROOT = DRIVE_ROOT / "PseudoKitchens_processed"
assert (DRIVE_KITCHENS_ROOT / "manifest.csv").is_file(), (
    f"manifest.csv tidak ditemukan: {DRIVE_KITCHENS_ROOT}. Jalankan notebook setup/konversi PseudoKitchens dahulu."
)
assert (DRIVE_KITCHENS_ROOT / "info.json").is_file(), "info.json PseudoKitchens tidak ditemukan."

WORK_ROOT = Path("/content/HiProbCBM/datasets/PseudoKitchens")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)
if WORK_ROOT.exists() or WORK_ROOT.is_symlink():
    assert WORK_ROOT.is_symlink() and WORK_ROOT.resolve() == DRIVE_KITCHENS_ROOT.resolve(), (
        f"Symlink PseudoKitchens tidak sesuai: {WORK_ROOT}. Jangan hapus folder non-symlink secara otomatis."
    )
else:
    os.symlink(DRIVE_KITCHENS_ROOT, WORK_ROOT, target_is_directory=True)
print("PseudoKitchens siap:", WORK_ROOT, "->", DRIVE_KITCHENS_ROOT)


## 4. Sambungkan train_log ke Drive dan cek GPU

In [ ]:
TRAIN_LOG_DRIVE = DRIVE_ROOT / "HiProbCBM_results" / "train_log"
TRAIN_LOG_DRIVE.mkdir(parents=True, exist_ok=True)
TRAIN_LOG_LOCAL = Path("/content/HiProbCBM/train_log")
if TRAIN_LOG_LOCAL.exists() or TRAIN_LOG_LOCAL.is_symlink():
    assert TRAIN_LOG_LOCAL.is_symlink() and TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), (
        f"train_log lokal bukan symlink yang diharapkan: {TRAIN_LOG_LOCAL}"
    )
else:
    os.symlink(TRAIN_LOG_DRIVE, TRAIN_LOG_LOCAL, target_is_directory=True)

def assert_persistent_training_storage():
    assert Path("/content/drive/MyDrive").is_dir(), "Google Drive belum mounted."
    assert TRAIN_LOG_LOCAL.is_symlink(), "train_log harus symlink ke Drive."
    assert TRAIN_LOG_LOCAL.resolve() == TRAIN_LOG_DRIVE.resolve(), "Target train_log berubah."

assert_persistent_training_storage()
assert torch.cuda.is_available(), "GPU belum aktif. Pilih runtime GPU sebelum training."
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint Drive:", TRAIN_LOG_DRIVE)


## 5. Analisis tambahan untuk delapan skenario

In [ ]:
import csv
import json
import statistics
from hiprobcbm.config import Config, load_config, prepare_log_dir, resolve_device
from hiprobcbm.engine.analysis import (evaluate_discovered_subconcepts, evaluate_uncertainty,
                                       summarize_intervention_order)
from hiprobcbm.utils.seed import set_random_seed

# (notebook asal, PROTOKOL, VARIAN, config Tahap 1, config Tahap 2)
SKENARIO = [
    ("01", "probcbm", "utama", "configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
    ("02", "probcbm", "a1", "configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
    ("03", "probcbm", "a2", "configs/cub_stage1_probcbm_reported.yaml", "configs/cub_stage2_probcbm_reported.yaml"),
    ("04", "probcbm", "a3", "configs/cub_stage1_probcbm_reported_batchtopk.yaml", "configs/cub_stage2_probcbm_reported_batchtopk.yaml"),
    ("05", "hicem", "utama", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
    ("06", "hicem", "a1", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
    ("07", "hicem", "a2", "configs/kitchens_stage1_hicem_reported.yaml", "configs/kitchens_stage2_hicem_reported.yaml"),
    ("08", "hicem", "a3", "configs/kitchens_stage1_hicem_reported_batchtopk.yaml", "configs/kitchens_stage2_hicem_reported_batchtopk.yaml"),
]
SEEDS = [42, 43, 44]

def stage1_dir(seed):
    induk = "a3" if VARIAN == "a3" else "utama"  # A1/A2 memakai hierarchy utama
    return prepare_log_dir(load_config(STAGE1_CONFIG), f"train_log/{PROTOKOL}/{induk}/seed_{seed}")

def stage2_dir(seed):
    values = load_config(STAGE2_CONFIG).to_dict()
    if VARIAN in {"a1", "a2"}:
        values["experiment_name"] = f"{values['experiment_name']}_ablation_{VARIAN}"
    return prepare_log_dir(Config(values), f"train_log/{PROTOKOL}/{VARIAN}/seed_{seed}")

def scalars(unc, extra):
    row = {}
    if isinstance(unc["cutout"], dict):
        row["cutout_concept_increase_own"] = unc["cutout"]["concept_increase_own"]
        row["cutout_concept_increase_propagated"] = unc["cutout"]["concept_increase_propagated"]
        row["cutout_class_increase"] = unc["cutout"]["class_increase"]
    for key, name in (("concept_groups_own", "concept_own"), ("concept_groups_propagated", "concept_propagated"),
                      ("class_groups", "class")):
        row[f"{name}_spearman"] = unc[key]["spearman_uncertainty_accuracy"]
        row[f"{name}_acc_lowest_unc"] = unc[key]["accuracy_lowest_uncertainty_group"]
        row[f"{name}_acc_highest_unc"] = unc[key]["accuracy_highest_uncertainty_group"]
    row.update(extra)
    return row

assert_persistent_training_storage()
device = resolve_device(0)
all_rows = []
for NOTEBOOK, PROTOKOL, VARIAN, STAGE1_CONFIG, STAGE2_CONFIG in SKENARIO:
    print(f"\n{'=' * 72}\nNotebook {NOTEBOOK}: {PROTOKOL} / {VARIAN}\n{'=' * 72}")
    rows = []
    for seed in SEEDS:
        parent_dir, child_dir = stage1_dir(seed), stage2_dir(seed)
        best = child_dir / "stage2_best.pth"
        manifest = child_dir / "run_manifest.json"
        hierarchy_path = parent_dir / "pseudo_hierarchy.pt"
        if not (best.exists() and manifest.exists() and hierarchy_path.exists()):
            print(f"LEWATI seed {seed}: checkpoint atau hierarchy belum lengkap.")
            continue
        cfg = Config(json.loads(manifest.read_text(encoding="utf-8"))["config"])
        counts = torch.load(hierarchy_path, map_location="cpu", weights_only=True)["subconcepts_per_concept"]

        set_random_seed(cfg.get("seed", seed))
        unc = evaluate_uncertainty(cfg, best, counts, device)
        (child_dir / "uncertainty_analysis.json").write_text(json.dumps(unc, indent=2), encoding="utf-8")
        extra = {}
        if PROTOKOL == "probcbm":
            curves = json.loads((child_dir / "intervention_curves.json").read_text(encoding="utf-8"))["curves"]
            order = summarize_intervention_order(curves)
            extra = {"intervention_area_uncertainty_order": order["area_uncertainty"],
                     "intervention_area_random_order": order["area_random"]}
        else:
            set_random_seed(cfg.get("seed", seed))
            disc = evaluate_discovered_subconcepts(cfg, best, hierarchy_path, device)
            (child_dir / "discovered_subconcepts.json").write_text(json.dumps(disc, indent=2), encoding="utf-8")
            extra = {"discovered_n_matched": disc["n_matched"],
                     "discovered_n_ground_truth": disc["n_ground_truth_subconcepts"],
                     "discovered_mean_roc_auc": disc["mean_discovered_concept_roc_auc"],
                     "discovered_intervention_delta": disc["intervention_delta"]}
        row = {"notebook": NOTEBOOK, "protokol": PROTOKOL, "varian": VARIAN, "seed": seed, **scalars(unc, extra)}
        rows.append(row)
        print(f"OK seed {seed}: {row}")
    all_rows.extend(rows)
    for metric in [k for k in (rows[0] if rows else {}) if k not in ("notebook", "protokol", "varian", "seed")]:
        values = [r[metric] for r in rows if isinstance(r.get(metric), (int, float)) and r[metric] == r[metric]]
        if values:
            std = statistics.stdev(values) if len(values) > 1 else 0.0
            print(f"{metric}: {statistics.mean(values):.4f} +/- {std:.4f} (n={len(values)})")

out_csv = Path("/content/HiProbCBM") / "train_log" / "analisis_tambahan.csv"
if all_rows:
    fields = list(dict.fromkeys(k for r in all_rows for k in r))
    with out_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(all_rows)
    print("\nCSV:", out_csv)
